# 02 · Construir un servidor MCP con `fastmcp`

En el notebook anterior usamos `mcp.server.fastmcp.FastMCP` con transporte stdio.
Ahora vamos a usar el paquete **`fastmcp`** (superset del anterior, con más utilidades)
con transporte **`streamable-http`**: un servidor que escucha en un puerto TCP y puede
atender múltiples clientes — el mismo patrón que usa `jub-agent/mcp/server.py`.

El código real que vamos a correr **ya está empaquetado** en `../mcp_server/` (no lo
reescribimos aquí): este notebook lo explica y lo ejercita.

Requisitos: `pip install fastmcp python-dotenv "mcp[cli]"`.


## El patrón `register(mcp)`

Abre [`../mcp_server/tools/basic.py`](../mcp_server/tools/basic.py). Cada módulo de
tools expone una función `register(mcp)` que define las tools como funciones internas
decoradas con `@mcp.tool()`:

```python
def register(mcp) -> None:
    @mcp.tool()
    def sumar(a: float, b: float) -> float:
        """Suma dos números y devuelve el resultado."""
        return a + b

    @mcp.tool()
    def crear_nota(texto: str) -> dict:
        """Crea una nota nueva con el texto dado y la guarda en memoria..."""
        ...
```

Y [`../mcp_server/server.py`](../mcp_server/server.py) simplemente crea el `FastMCP` y
llama `register(mcp)` de cada módulo:

```python
mcp = FastMCP(name="tutorial-mcp", instructions="...")
basic.register(mcp)
catalog.register(mcp)  # lo exploramos en el notebook 05

mcp.run(transport="streamable-http", host="0.0.0.0", port=MCP_PORT)
```

Este es exactamente el mismo patrón que `jub-agent/mcp/server.py`: un módulo por
dominio de tools, cada uno auto-contenido, registrado explícitamente (sin
auto-discovery mágico — más fácil de leer y depurar).


In [ ]:
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

MCP_SERVER_DIR = Path("..") / "mcp_server"
MCP_PORT = 8100  # puerto dedicado para este notebook (el docker-compose usa 8000)

env = os.environ.copy()
env["MCP_PORT"] = str(MCP_PORT)

server_process = subprocess.Popen(
    [sys.executable, "server.py"],
    cwd=MCP_SERVER_DIR,
    env=env,
)


def esperar_puerto(host, port, timeout=30):
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f"El servidor no abrió el puerto {port} a tiempo")


esperar_puerto("localhost", MCP_PORT)
print(f"Servidor MCP arriba en http://localhost:{MCP_PORT}/mcp (pid={server_process.pid})")




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                                                              │
│                                FastMCP 3.4.7                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      tutorial-mcp, 3.4.7                         │
│                  🚀 Deplo

Servidor MCP arriba en http://localhost:8100/mcp (pid=76341)


INFO:     127.0.0.1:53358 - "GET /mcp HTTP/1.1" 406 Not Acceptable
INFO:     127.0.0.1:53358 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50674 - "GET /docs HTTP/1.1" 404 Not Found


## Hablarle con el cliente MCP crudo (sin agentes todavía)

Igual que en el notebook 01, pero ahora sobre HTTP en vez de stdio, usando
`streamablehttp_client` del SDK oficial `mcp`.


In [2]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_URL = f"http://localhost:{MCP_PORT}/mcp"


async def probar_tools_basicas():
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Tools disponibles:", [t.name for t in tools.tools])

            r = await session.call_tool("sumar", {"a": 4, "b": 5})
            print("sumar(4, 5) ->", r.content[0].text)

            r = await session.call_tool("crear_nota", {"texto": "comprar café"})
            print("crear_nota(...) ->", r.content[0].text)

            r = await session.call_tool("listar_notas", {})
            print("listar_notas() ->", r.content[0].text)


await probar_tools_basicas()


INFO:     127.0.0.1:53740 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:53746 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:53742 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:53750 - "POST /mcp HTTP/1.1" 200 OK
Tools disponibles: ['sumar', 'multiplicar', 'crear_nota', 'listar_notas', 'completar_nota', 'list_catalogo', 'get_item', 'search_catalogo', 'resumen_por_tipo']
INFO:     127.0.0.1:53760 - "POST /mcp HTTP/1.1" 200 OK
sumar(4, 5) -> 9.0
INFO:     127.0.0.1:53774 - "POST /mcp HTTP/1.1" 200 OK
crear_nota(...) -> {"id":1,"texto":"comprar café","completada":false}
INFO:     127.0.0.1:53778 - "POST /mcp HTTP/1.1" 200 OK
listar_notas() -> [{"id":1,"texto":"comprar café","completada":false}]
INFO:     127.0.0.1:53782 - "DELETE /mcp HTTP/1.1" 200 OK


Nota algo importante: el estado de las notas vive **en el proceso del servidor**,
no en el notebook ni en el cliente. Si vuelves a correr `listar_notas` verás que las
notas persisten mientras el servidor siga corriendo — así es como funcionaría en
producción, con múltiples clientes (agentes, otros scripts) compartiendo el mismo
estado a través del servidor MCP.

**Siguiente:** en [`03_agent_framework_basico.ipynb`](03_agent_framework_basico.ipynb)
dejamos el cliente MCP crudo a un lado y conocemos `agent-framework`, todavía sin MCP.
Luego, en el notebook 04, los juntamos.


In [3]:
# Limpieza: apaga el servidor MCP de este notebook antes de pasar al siguiente.
server_process.terminate()
server_process.wait(timeout=5)
print("Servidor detenido.")


Servidor detenido.


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [76341]
